# 05 — Run Comparison And mAP Action Plan

Goal: decide what to try next from evidence, not guesses.

Use this after several training runs. It reads `runs/detect/*/results.csv`, compares validation metrics, and creates an action plan checklist.

What to look for:

- **Best epoch much earlier than final epoch**: possible overfitting or too-high learning rate.
- **mAP still climbing at final epoch**: train longer.
- **mAP50 high but Kaggle low**: public test distribution or threshold/submission issue.
- **One class AP much lower**: inspect data/labels for that class before changing global hyperparameters.
- **Better local mAP but worse Kaggle**: do not overfit to validation; inspect split mismatch.


In [ ]:
from pathlib import Path
import os
import random
import sys


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "config" / "competition.yaml").is_file():
            return path
    raise FileNotFoundError("Could not find repo root containing config/competition.yaml")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print(REPO_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUNS_ROOT = REPO_ROOT / "runs" / "detect"
EXPERIMENT_LOG = REPO_ROOT / "experiments" / "experiment_log.csv"
SUBMISSION_LOG = REPO_ROOT / "experiments" / "submission_log.csv"

print("runs root:", RUNS_ROOT, RUNS_ROOT.is_dir())
print("experiment log:", EXPERIMENT_LOG, EXPERIMENT_LOG.is_file())
print("submission log:", SUBMISSION_LOG, SUBMISSION_LOG.is_file())


## Load Training Curves

Comments:

- Ultralytics column names vary by version; this cell searches for likely mAP columns.
- If a run has no `results.csv`, it did not finish or outputs were moved.
- Compare run names to `config/competition.yaml` and experiment logs to avoid mixing data revisions.


In [ ]:
def clean_columns(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame.columns = [str(col).strip() for col in frame.columns]
    return frame


def find_metric_column(columns: list[str], candidates: list[str]) -> str | None:
    lowered = {col.lower(): col for col in columns}
    for candidate in candidates:
        for low, original in lowered.items():
            if candidate.lower() in low:
                return original
    return None


run_frames = []
summary_rows = []
for results_csv in sorted(RUNS_ROOT.glob("*/results.csv")) if RUNS_ROOT.is_dir() else []:
    run_name = results_csv.parent.name
    frame = clean_columns(pd.read_csv(results_csv))
    frame["run_name"] = run_name
    frame["epoch_index"] = np.arange(len(frame))
    run_frames.append(frame)

    map50_col = find_metric_column(list(frame.columns), ["metrics/map50(b)", "map50"])
    map5095_col = find_metric_column(list(frame.columns), ["metrics/map50-95(b)", "map50-95", "map"])
    precision_col = find_metric_column(list(frame.columns), ["metrics/precision"])
    recall_col = find_metric_column(list(frame.columns), ["metrics/recall"])

    row = {"run_name": run_name, "epochs_completed": len(frame), "results_csv": str(results_csv)}
    for label, col in [("map50", map50_col), ("map50_95", map5095_col), ("precision", precision_col), ("recall", recall_col)]:
        if col is not None:
            values = pd.to_numeric(frame[col], errors="coerce")
            best_idx = int(values.idxmax()) if values.notna().any() else None
            row[f"best_{label}"] = float(values.max()) if values.notna().any() else np.nan
            row[f"best_{label}_epoch"] = best_idx + 1 if best_idx is not None else np.nan
            row[f"final_{label}"] = float(values.iloc[-1]) if len(values) else np.nan
            row[f"{label}_column"] = col
    summary_rows.append(row)

runs = pd.concat(run_frames, ignore_index=True) if run_frames else pd.DataFrame()
run_summary = pd.DataFrame(summary_rows).sort_values("best_map50", ascending=False) if summary_rows else pd.DataFrame()
display(run_summary)


In [ ]:
if runs.empty or run_summary.empty or "map50_column" not in run_summary.columns:
    print("No run curves available yet.")
else:
    plt.figure(figsize=(12, 5))
    for _, row in run_summary.head(8).iterrows():
        run_name = row["run_name"]
        col = row.get("map50_column")
        if not isinstance(col, str):
            continue
        frame = runs[runs["run_name"] == run_name]
        plt.plot(frame["epoch_index"] + 1, pd.to_numeric(frame[col], errors="coerce"), label=run_name)
    plt.title("Validation mAP@0.5 by run")
    plt.xlabel("epoch")
    plt.ylabel("mAP@0.5")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()


## Diagnose Training Behavior

Use this table to choose the next training experiment.

- `best_epoch_fraction < 0.4`: run may peak early; try lower `lr0`, less augmentation, or inspect overfitting.
- `best_epoch_fraction > 0.85`: run may need more epochs.
- `final_gap` strongly negative: final model worse than best; make sure prediction uses `best.pt`, not `last.pt`.


In [ ]:
if run_summary.empty or "best_map50_epoch" not in run_summary.columns:
    print("Need at least one finished run with mAP metrics.")
else:
    diagnostics = run_summary.copy()
    diagnostics["best_epoch_fraction"] = diagnostics["best_map50_epoch"] / diagnostics["epochs_completed"]
    diagnostics["final_gap"] = diagnostics["final_map50"] - diagnostics["best_map50"]
    diagnostics["suggested_next_step"] = np.select(
        [
            diagnostics["best_epoch_fraction"] > 0.85,
            diagnostics["best_epoch_fraction"] < 0.40,
            diagnostics["best_map50"] < diagnostics["best_map50"].max() - 0.03,
        ],
        [
            "train longer or reduce early stopping pressure",
            "try lower lr0 or inspect overfitting/augmentation",
            "compare config/data revision to best run",
        ],
        default="candidate for prediction threshold and error analysis",
    )
    display(diagnostics[["run_name", "best_map50", "best_map50_epoch", "epochs_completed", "best_epoch_fraction", "final_gap", "suggested_next_step"]].round(4))


## Join Experiment And Submission Logs

Comments:

- Local mAP should decide most iterations; Kaggle public score should validate direction.
- If logs are empty, start filling them now. Winning requires traceability.
- If a submission is not linked to an experiment, it is hard to learn from it.


In [ ]:
experiment_log = pd.read_csv(EXPERIMENT_LOG) if EXPERIMENT_LOG.is_file() else pd.DataFrame()
submission_log = pd.read_csv(SUBMISSION_LOG) if SUBMISSION_LOG.is_file() else pd.DataFrame()

if experiment_log.empty:
    print("experiment_log.csv has no rows yet.")
else:
    display(experiment_log.tail(10))

if submission_log.empty:
    print("submission_log.csv has no rows yet.")
else:
    display(submission_log.tail(10))

if not experiment_log.empty and not submission_log.empty and "experiment_id" in submission_log:
    joined = submission_log.merge(experiment_log, on="experiment_id", how="left", suffixes=("_submission", "_experiment"))
    display(joined.tail(10))


## Evidence-Based Action Plan

Fill these after running notebooks `03` and `04`. The generated checklist should drive the next 1–3 experiments.


In [ ]:
# Fill these manually from notebooks 03 and 04.
OBSERVED = {
    "dominant_failure": "",  # false_negative, background_or_missing_label, localization_error, class_confusion, unknown
    "weakest_class": "",     # truck, car, van, bus
    "label_issue_bucket": "", # tiny, huge, extreme_aspect, edge_touch, crowded, none
    "best_run_still_climbing": False,
    "public_score_below_local_expectation": False,
}

rules = []
if OBSERVED["dominant_failure"] == "false_negative":
    rules.append((1, "Review false negatives in 3LC, especially visible missed vehicles; then train longer or lower conf."))
if OBSERVED["dominant_failure"] == "background_or_missing_label":
    rules.append((1, "Inspect high-confidence background FPs; if they are real vehicles, fix missing labels and retrain."))
if OBSERVED["dominant_failure"] == "localization_error":
    rules.append((1, "Review loose/tight boxes and tiny objects; fix labels before tuning lr."))
if OBSERVED["dominant_failure"] == "class_confusion":
    rules.append((1, f"Audit class policy and labels for {OBSERVED['weakest_class'] or 'confused classes'}."))
if OBSERVED["label_issue_bucket"]:
    rules.append((2, f"Create a 3LC filter/revision for label bucket: {OBSERVED['label_issue_bucket']}"))
if OBSERVED["best_run_still_climbing"]:
    rules.append((2, "Run a longer training config: 40/60/80 epochs with the same data revision."))
if OBSERVED["public_score_below_local_expectation"]:
    rules.append((3, "Check validation/test distribution mismatch and sweep predict.conf on validation before next Kaggle submit."))

if not rules:
    rules.append((1, "Run notebooks 03 and 04, then fill OBSERVED to generate a concrete action plan."))

action_plan = pd.DataFrame(rules, columns=["priority", "action"]).sort_values("priority")
display(action_plan)


## Recommended Next Experiments Once Data Issues Are Addressed

Keep the experiment grid small and interpretable:

1. Same data revision, train longer: `epochs=40`, `lr0=0.005`.
2. Same data revision, train longer: `epochs=60`, `lr0=0.002`.
3. Compare augmentation after label cleanup: `use_augmentation=true` vs `false`.
4. After choosing best weights, sweep prediction thresholds on validation: `conf=0.10/0.15/0.20/0.25`, `iou=0.50/0.60/0.70`.

Do not submit every run to Kaggle. Submit only candidates with a clear local or data-quality reason.
